## 1. Load and Inspect Silver GDP Data

Load annual GDP data from the Silver layer and inspect the schema, total record count, and sample values before building the Gold table.

In [ ]:
df_gdp = spark.table("silver_worldbank_gdp")

df_gdp.printSchema()
print("Total rows:", df_gdp.count())
df_gdp.show(10, truncate=False)

## 2. Validate GDP Data Quality

Validate the annual GDP dataset by checking:
- Duplicate country-year combinations.
- Record counts and year ranges for each country.
- NULL values and non-positive GDP values.

In [ ]:
from pyspark.sql import functions as F

df_gdp.groupBy("country", "year") \
    .count() \
    .filter(F.col("count") > 1) \
    .orderBy(F.col("year").desc()) \
    .show(10, truncate=False)

In [ ]:
df_gdp.groupBy("country").agg(
    F.count("*").alias("total_rows"),
    F.min("year").alias("min_year"),
    F.max("year").alias("max_year")
).show(truncate=False)

In [ ]:
df_gdp.select(
    F.sum(F.col("country").isNull().cast("int")).alias("null_country"),
    F.sum(F.col("year").isNull().cast("int")).alias("null_year"),
    F.sum(F.col("gdp_usd").isNull().cast("int")).alias("null_gdp"),
    F.sum((F.col("gdp_usd") <= 0).cast("int")).alias("invalid_gdp")
).show()

## 3. Build, Save and Verify Gold GDP

Prepare the annual GDP dataset with consistent data types, save it as a Gold Delta table, and verify the persisted schema and records.

In [ ]:
df_gold_gdp = df_gdp.select(
    F.col("country"),
    F.col("year").cast("int").alias("year"),
    F.col("gdp_usd")
)

df_gold_gdp.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_gdp_annual")

In [ ]:
df_check = spark.table("gold_gdp_annual")

df_check.printSchema()
print("Total rows:", df_check.count())
df_check.orderBy(F.col("year").desc()).show(6, truncate=False)